In dit notebook wordt uitgewerkt hoe je topics kunt verwijderen.

In [1]:
from ask_delphi_api.import_digicoach import Import

import_service = Import()

AskDelphi Client loaded.
Parsed tenant/project/acl from CMS URL
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


In [2]:
def _delete_topic_relation(source_id: str, source_version_id: str, target_id: str, relation_type_id: str):
    """POST-call om een topic-relatie te verwijderen."""
    endpoint = (
        f"v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}"
        f"/topic/{source_id}/topicVersion/{source_version_id}/relation/delete"
    )
    data = {
        "relationTypeId": relation_type_id,
        "sourceTopicId": source_id,
        "targetTopicId": target_id,
    }
    return import_service.client._request("POST", endpoint, json_data=data)


def delete_relation(topic_id_source: str, topic_id_target: str, relation_name: str):
    """
    Verwijder een relatie van source → target van het type `relation_name`.
    Doet: checkout → resolve relation_type_id → delete → checkin → publiceer.
    """
    response = None
    try:
        import_service.topic.checkout(topic_id_source)
        source_version_id = import_service.topic.get_topicVersionId(topic_id_source)

        relation_type_id = import_service.relation.get_relation_type_id(
            topic_id_source, source_version_id, relation_name
        )

        response = _delete_topic_relation(
            topic_id_source, source_version_id, topic_id_target, relation_type_id
        )

    except Exception as e:
        print(f"Opruimen van topic relation mislukt: {e}")
    finally:
        # Deze twee moeten altijd gebeuren om de state netjes te houden
        try:
            import_service.topic.checkin(topic_id_source)
        finally:
            import_service.publiceer(topic_id_source)

    return response

def get_topic_relation(topicId: str):
    """Opvragen topic relations."""
    endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topicId}/relation"
    data = {}
    return import_service.client._request("GET", endpoint, json_data=data)

def delete_topic(topic_id: str, version_id: str, workflowstage_ids: list):
    """Voert de DELETE-call uit voor een topic."""
    endpoint = (
        f"v3/tenant/{{tenantId}}/project/{{projectId}}/"
        f"acl/{{aclEntryId}}/topic/{topic_id}/topicVersion/{version_id}"
    )
    data = {"workflowActions": {
        "applyWorkflowStageIds": workflowstage_ids,
        "increaseMajorVersionNo": True}
    }
    return import_service.client._request("DELETE", endpoint, json_data=data)


def soft_delete_topic(topic_id: str,  workflowstage_ids: list):
    """Soft delete: checkout → delete → checkin."""
    try:
        import_service.topic.checkout(topic_id)
        version_id = import_service.topic.get_topicVersionId(topic_id)
        response = delete_topic(topic_id, version_id, workflowstage_ids)
    except Exception as e:
        print(f"Soft delete mislukt: {e}")
        response = {}
    finally:
        import_service.topic.checkin(topic_id)

    return response

def get_topic_ids(items: list, targetTopicType: str):
    topic_ids =[]
    for d in items:
        if d.get("targetTopicType") == targetTopicType and d.get("targetTopicIsDeleted") is False :
            topic_ids.append(d["targetTopicId"])
            print(f"{d['targetTopicName']}, {d["targetTopicId"]}")
    return topic_ids

import re

# Regex: YYYY-MM-DD HH:MM:SS of met microseconden
TITLE_DATETIME_REGEX = re.compile(r"\b\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2}(?:\.\d+)?\b")

def has_datetime_in_title(title: str) -> bool:
    """Checkt of de titel een datetime bevat in het formaat YYYY-MM-DD HH:MM:SS(.microseconds)."""
    if not isinstance(title, str):
        return False
    return bool(TITLE_DATETIME_REGEX.search(title))

def filter_topics_with_title_datetime(topics: list[dict]) -> list[dict]:
    """
    Retourneert alleen die topics waarvoor de 'title' een datetime bevat.
    Laat verder de originele dicts intact.
    """
    if not isinstance(topics, list):
        return []
    return [t for t in topics if isinstance(t, dict) and has_datetime_in_title(t.get("title", ""))]

def filter_topics(topicTypeName: str):
    """
    Filtert alle topics op topicTypeName
    en geeft een lijst met dicts terug met alleen topicGuid, title en lastModificationDate.
    """

    topics = import_service.topic.fetch_topiclist()

    result = []

    for topic in topics:
        if not isinstance(topic, dict):
            continue

        if topic.get("topicTypeName") == topicTypeName:
            result.append({
                "topicGuid": topic.get("topicGuid"),
                "title": topic.get("title"),
                "lastModificationDate": topic.get("lastModificationDate")
            })

    return result

### Verwijderen Digicoach taken, stappen en relaties

In [3]:
import pprint

from typing import Any, Optional

def get_workflow_id_by_name(payload: Any, target_name: str = "Default workflow") -> Optional[str]:
    """
    Zoekt veilig naar het eerste item in payload['data'] met name == target_name.
    Geeft het ID (str) terug of None als niet gevonden of als de structuur afwijkt.
    """

    if not isinstance(payload, dict):
        return None

    items = payload.get('data')
    if not isinstance(items, list) or not items:
        return None

    for item in items:
        if isinstance(item, dict) and item.get('name') == target_name:
            return item.get('id')

    return None

def extract_stage_ids(payload):
    """
    Haalt de stage IDs op voor Concept, Test en Productie.
    Geeft een dict terug: {'Concept': <id>, 'Test': <id>, 'Productie': <id>}
    """

    # De drie stages waar we ID's van willen
    target_titles = {"Concept", "Test", "Productie"}

    result = {title: None for title in target_titles}

    # Veilig navigeren naar de stages
    stages = (
        payload.get("data", {})
               .get("stages", [])
    )

    if not isinstance(stages, list):
        return result  # lege dict, alles None

    # Door de stage-objecten lopen
    for stage in stages:
        if not isinstance(stage, dict):
            continue

        title = stage.get("title")
        stage_id = stage.get("id")

        if title in target_titles:
            result[title] = stage_id

    return result


def get_workflowstate_ids():
    workflowstate_ids = {}

    # Ophalen workflow_id
    endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/workflow/search"
    data = {}
    response = import_service.client._request("POST", endpoint, json_data=data)
    workflow_id = get_workflow_id_by_name(response, "Default workflow")

    # Ophalen workflowstate_ids Concept, Test, Productie
    endpoint = f"v1/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/workflow/{workflow_id}"
    data = {}
    response = import_service.client._request("GET", endpoint, json_data=data)
    workflowstate_ids = extract_stage_ids(response)

    return workflowstate_ids

def delete_digicoach(topic_id_digicoach: str):

    # Ophalen workflowstage_ids
    # workflowstage_ids = get_workflowstate_ids()
    # workflowstage_ids_list = list(workflowstage_ids.values())
    # print(f"workflowstage_ids_list : {workflowstage_ids_list}")

    # Ophalen van workflowstage ids gaat soms niet goed, voor nuu een lijstje met ids
    workflowstage_ids_list = ['9d3b3151-b543-4d6f-a9e5-18f4388753e2', 'ad70aeea-a938-469d-a6fb-493883ae982b', '45faa337-2499-4f06-a5c2-c919e4291bb2']

    response = get_topic_relation(topic_id_digicoach)

    # Ophalen topic ids taken
    task_topic_ids = get_topic_ids(response['relations'], "Task")
    print(f"task_topic_ids : {task_topic_ids}")

    for task_topic_id in task_topic_ids:

        # Ophalen topic ids van de stappen
        response = get_topic_relation(task_topic_id)
        action_topic_ids = get_topic_ids(response['relations'], "Action")
        print(f"action_topic_ids : {action_topic_ids}")

        # Verwijder stap topics en relations met de taken
        for action_topic_id in action_topic_ids:
            print(f"Verwijder stap topic {action_topic_id}")
            delete_relation(task_topic_id, action_topic_id, "Stap")
            soft_delete_topic(action_topic_id, workflowstage_ids_list)

        # Verwijder taak topic en relation met de digicoach
        delete_relation(topic_id_digicoach, task_topic_id, "Taak")
        soft_delete_topic(task_topic_id, workflowstage_ids_list)

    response = soft_delete_topic(topic_id_digicoach, workflowstage_ids_list)
    print(response)

### Verwijder Digicoach processen

In [6]:
print("Selectie digicoaches gebruikt voor ontwikkeling")
digicoach_topics = filter_topics("Digitale Coach Procespagina")
digicoach_topics_filtered = filter_topics_with_title_datetime(digicoach_topics)
pprint.pprint(digicoach_topics_filtered)

print("Opruimen digicoaches gebruikt voor ontwikkeling")
for digicoach_topic in digicoach_topics_filtered:    
    print(f"Opruimen : {digicoach_topic["title"]}")
    delete_digicoach(digicoach_topic["topicGuid"])


Selectie digicoaches gebruikt voor ontwikkeling
[]
Opruimen digicoaches gebruikt voor ontwikkeling


### Opruimen Digicoach op basis van topic id

In [5]:
delete_digicoach("77d3604a-db53-4499-964e-453e52c3c2fd")

task_topic_ids : []
{}
